In [ ]:
using LensFactory
using LensFactory.Constants
using CairoMakie

# Initialize a cosmology

In [ ]:
# Initialize default cosmology
cosmo = Cosmology.init_cosmology()

# Lens and source redshifts
zl = 0.5
zs = 1.5

# ADDs and distance ratio
Dol = Cosmology.angular_diameter_distance(cosmo, 0., zl)
Dls = Cosmology.angular_diameter_distance(cosmo, zl, zs)
Dos = Cosmology.angular_diameter_distance(cosmo, 0., zs)
adis = Dls/Dos

# Create a grid
Since in most of the astrophysical scenario (primarily strong lensing by galaxies or galaxy clusters), the Einstein angle is ~[0.1, 50] arcseconds, it was decided to have `ANGLE_ARCSECOND` as default grid unit.

In [ ]:
# Create image plane grid (default units are ANGLE_ARCSEC)
x, y = Lenses.get_meshgrid(5, 5, 0.01);

# Initialize a point mass lens and plot

In [ ]:
# Initialize an isolated point mass lens
lens = Lenses.init_PointLens(D_d = Dol, mass=1E12)

# Plot the image plane (by default both source and image plane are shown in the same panel)
fig, ax = Lenses.plot_image_plane(lens, x, y, adis)
display(fig)

# Point Mass Lens + External Shear
For multi-component lens models, we use composite lens model. All the lens models does not have to same type. For example, to simulate point mass lens in environment effect, we can create a composite lens with two components, `PointLens` and `ExternalEffectsLens`. **While using, the `ExternalEffects` lens, it is important to note that we should pass ($\kappa$, $\gamma$) values for a source at infinity (i.e., $D_{ds}/D_s$). This is because all lensing qunatities are later re-scaled by `adis` for a given source redshift.**

In [ ]:
lens = Lenses.init_CompositeLens([(lens=:PointLens, x_c=0.0, y_c=0.0, D_d=Dol, mass=1E12),
                                  (lens=:ExternalEffects, kappa=0.0, gamma=0.1, angle=45)])

# Plot the image plane
fig, ax = Lenses.plot_image_plane(lens, x, y, adis; source=(0.1, 0.1))
display(fig)

# Two point mass lenses
To construct a binary lens, we can again use the composite lens model. 

In [ ]:
# Initialize a two component lens
lens = Lenses.init_CompositeLens([(lens=:PointLens, D_d=Dol, x_c=0.0, y_c=0.0, mass=1E11),
                                  (lens=:PointLens, D_d=Dol, x_c=1.0, y_c=1.0, mass=1E11)])

# Plot the image plane (by default both source and image plane are shown in the same panel)
fig, ax = Lenses.plot_image_plane(lens, x, y, adis)
display(fig)

# N-point mass lens
The same composite lens can alo used to mimic a ensamble of N point mass lenses. **However, it is important to note that this default method is not optimized for microlensing by a bunch of point mass lenses.**

In [ ]:
# To control the positions of point mass lenses
using Random
Random.seed!(1234)

# Initialize a lens made of multiple point mass lenses
n_point = 15
ensamble = [(lens=:PointLens, D_d=Dol, x_c=(-3.0 + 6.0*rand()), y_c = (-3.0 + 6.0*rand()), mass=1E11) for _ in 1:n_point]
lens = Lenses.init_CompositeLens(ensamble)

# Here we have split the source and image plane for better visualization using "two_panel" keyword
fig, ax = Lenses.plot_image_plane(lens, x, y, adis; two_panel=true, figure_size=(800, 400))
display(fig)
